In [1]:
from dotenv import load_dotenv
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
    reset_activity_context,
    visualize_conversation,
)

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient

load_dotenv()

# Define the model to use throughout this notebook
# Using Opus 4.6 for its superior planning and reasoning capabilities
MODEL = "claude-opus-4-6"
print(f"📋 Notebook configured to use: {MODEL}")

📋 Notebook configured to use: claude-opus-4-6


# 01 - Chief of Staff 에이전트

#### 들어가며

노트북 00에서 간단한 리서치 에이전트를 만들었습니다. 이 노트북에서는 포괄적인 에이전트를 만들기 위한 Claude Code SDK의 핵심 기능들을 하나씩 소개합니다. 기능마다 다음을 설명합니다.
- **무엇(What)**: 그 기능이 무엇인지
- **왜(Why)**: 그 기능으로 무엇을 할 수 있고 왜 쓰고 싶은지
- **어떻게(How)**: 사용법을 보여 주는 최소 구현

Claude Code에 익숙하다면, SDK가 동일한 기능을 제공해 Claude Code의 모든 역량을 프로그램 방식의 헤드리스 환경에서 활용할 수 있게 해 준다는 점을 알아채실 것입니다.

#### 시나리오

이 노트북 전반에서 방금 시리즈 A로 1천만 달러를 유치한 50인 규모 스타트업을 위한 **AI Chief of Staff**를 만듭니다. CEO는 공격적인 성장과 재무 지속 가능성 사이의 균형을 잡기 위해 데이터 기반 통찰이 필요합니다.

최종 Chief of Staff 에이전트는 다음을 수행합니다.
- 영역별 **전문 서브에이전트 조율**
- 여러 출처의 **통찰 종합**
- 실행 가능한 권고가 담긴 **경영진용 요약 제공**

## 기본 기능

### 기능 0: [CLAUDE.md](https://www.anthropic.com/engineering/claude-code-best-practices)를 통한 메모리

**무엇**: `CLAUDE.md` 파일은 에이전트를 위한 지속적인 메모리이자 지시문 역할을 합니다. 프로젝트 디렉터리에 있으면 Claude Code가 에이전트를 초기화할 때 이 맥락을 자동으로 읽어 반영합니다.

**왜**: 상호작용할 때마다 프로젝트 맥락, 팀 선호, 기준을 반복해 제공하는 대신 `CLAUDE.md`에 한 번 정의해 두면 됩니다. 일관된 동작을 보장하고 중복 설명을 없애 토큰 사용도 줄여 줍니다.

**어떻게**: 
- 작업 디렉터리에 `CLAUDE.md` 파일을 둡니다. 이 예제에서는 `chief_of_staff_agent/CLAUDE.md`입니다
- ClaudeSDKClient의 `cwd` 인자가 CLAUDE.md가 있는 디렉터리를 가리키게 합니다
- 세부 데이터 파일보다 상위 맥락을 우선하기를 원한다면 명시적인 프롬프트로 에이전트를 이끄세요

**동작에 대한 중요한 참고**: CLAUDE.md와 세부 데이터 파일(CSV 등)이 모두 있을 때, 에이전트는 정확한 답을 주기 위해 더 세밀한 데이터 출처를 읽으려 할 수 있습니다. 예상된 동작입니다. 에이전트는 자연스럽게 권위 있는 데이터를 찾습니다. 에이전트가 상위 수준의 CLAUDE.md 맥락을 쓰게 하려면 명시적인 프롬프트 지시를 사용하세요(아래 예 참고). 여기서 중요한 교훈을 얻습니다. CLAUDE.md는 데이터 출처에 대한 강한 제약이 아니라 *맥락과 안내*를 제공한다는 것입니다.

In [2]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",  # Points to subdirectory with our CLAUDE.md
        setting_sources=["project"],
    )
) as agent:
    await agent.query("What's our current runway?")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# Display the response with HTML rendering
display_agent_response(messages)
# With this prompt, the agent should use CLAUDE.md values: ~$500K burn, 20 months runway

🤖 Thinking...


#### 에이전트의 데이터 출처 선호 이해하기

**방금 무슨 일이 있었나:**
프롬프트에 해당 문구를 추가함으로써, 에이전트가 CSV 파일에서 더 세밀한 데이터를 찾는 대신 CLAUDE.md 맥락에 의존하도록 이끌었습니다.

**핵심 통찰:**
1. **제약이 아니라 맥락으로서의 CLAUDE.md**: `cwd`를 설정하면 CLAUDE.md 파일이 배경 맥락으로 적재됩니다. 하지만 에이전트는 자연스럽게 가장 권위 있는 데이터 출처를 찾습니다. 세부 CSV 파일이 있으면 정확성을 위해 그쪽을 선호할 수 있습니다.

2. **프롬프트 엔지니어링이 중요합니다**: "맥락에 있는 상위 수준 재무 수치"라는 표현은 financial_data/burn_rate.csv의 월별 정밀 데이터(총 소진 $525K, 순 소진 $235K)가 아니라 CLAUDE.md의 간략한 경영진 요약(소진 $500K, 런웨이 20개월)을 원한다는 신호를 에이전트에 보냅니다.

3. **아키텍처 설계 선택**: 이 동작은 사실 프로덕션 시스템에서 바람직합니다. 에이전트가 최선의 데이터 출처를 찾기를 바라니까요. CLAUDE.md에는 다음이 담겨야 합니다.
   - 상위 수준의 맥락과 전략
   - 회사 정보와 기준
   - 세부 데이터가 어디에 있는지 알려 주는 안내
   - 상위 수치와 세부 수치를 각각 언제 쓸지에 대한 지침

4. **현실의 패턴**: CLAUDE.md는 에이전트의 방향을 잡아 주는 "온보딩 문서"이고, 세부 파일은 정확성이 중요할 때 에이전트가 질의하는 "원천 시스템"이라고 생각하세요.

### 기능 1: Python 스크립트 실행을 위한 Bash 도구

**무엇**: Bash 도구를 사용하면 에이전트가 (여러 일 중에서도) Python 스크립트를 직접 실행할 수 있어, 절차적 지식, 복잡한 계산, 데이터 분석 등 에이전트의 기본 역량을 넘어서는 통합에 접근할 수 있습니다.

**왜**: Chief of Staff는 데이터 파일을 처리하거나, 재무 모델을 돌리거나, 그 데이터로 시각화를 만들어야 할 수 있습니다. 모두 Bash 도구를 쓰기 좋은 상황입니다.

**어떻게**: 에이전트가 닿을 수 있는 곳에 Python 스크립트를 두고, 그것이 무엇이며 어떻게 호출하는지 맥락을 덧붙이세요. 스크립트가 Chief of Staff 에이전트를 위한 것이라면 그 맥락을 CLAUDE.md에, 서브에이전트를 위한 것이라면 해당 MD 파일에 추가하세요(자세한 내용은 뒤에서 다룹니다). 이 튜토리얼에서는 `chief_of_staff_agent/scripts`에 예제 다섯 개를 넣어 두었습니다.
1. `hiring_impact.py`: 신규 엔지니어 채용이 소진율, 런웨이, 현금 잔고에 미치는 영향을 계산합니다. `financial-analyst` 서브에이전트가 월 $500K 소진과 20개월 런웨이를 기준으로 채용 시나리오를 모델링하는 데 필수적입니다.
2. `talent_scorer.py`: 가중 기준으로 후보자의 기술력, 경력, 문화 적합성, 희망 연봉을 점수화합니다. `recruiter` 서브에이전트가 TechStart의 시니어 엔지니어 기준인 $180~220K에 비추어 후보를 순위 매기는 핵심 도구입니다.
3. `simple_calculation.py`: 런웨이, 소진율, 분기 지표에 대한 빠른 재무 계산을 수행합니다. 복잡한 모델링 없이 즉시 지표를 얻기 위한 Chief of Staff용 유틸리티 스크립트입니다.
4. `financial_forecast.py`: 현재 월 15% 성장 중인 $2.4M ARR을 바탕으로 ARR 성장 시나리오(기본/낙관/비관)를 모델링합니다. `financial-analyst`가 시리즈 B 준비도를 예측하고 $30M 조달 목표를 검증하는 데 결정적입니다.
5. `decision_matrix.py`: SmartDev 인수나 사무실 확장 같은 전략적 선택을 위한 가중 의사결정 매트릭스를 만듭니다. 이해관계자와 기준이 여럿인 복잡한 결정을 Chief of Staff가 체계적으로 평가하도록 돕습니다.

In [3]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        allowed_tools=["Bash", "Read"],
        cwd="chief_of_staff_agent",  # Points to subdirectory where our agent is defined
    )
) as agent:
    await agent.query(
        "Use your simple calculation script with a total runway of 2904829 and a monthly burn of 121938."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# Display the response with HTML rendering
display_agent_response(messages)

🤖 Using: Glob()
🤖 Using: Glob()
🤖 Using: Glob()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: Bash()
✓ Tool completed
🤖 Thinking...


Metric,Value
Total Runway,"$2,904,829.00"
Monthly Burn,"$121,938.00"
Runway Months,~23.82 months
Quarterly Burn,"$365,814.00"
Daily Burn Rate,"$4,064.60"


### 기능 2: 출력 스타일

**무엇**: 출력 스타일을 사용하면 청중에 따라 서로 다른 출력 형식을 쓸 수 있습니다. 각 스타일은 마크다운 파일로 정의합니다.

**왜**: 에이전트를 쓰는 사람들의 전문성 수준이나 우선순위가 다를 수 있습니다. 출력 스타일을 쓰면 별도의 에이전트를 만들지 않고도 이런 층위를 구분할 수 있습니다.

**어떻게**:
- `chief_of_staff_agent/.claude/output-styles/`에 스타일마다 마크다운 파일을 만드세요. 예로 `.claude/output-styles/executive.md`의 경영진 출력 스타일을 확인해 보세요. 출력 스타일은 name과 description 두 필드로 이뤄진 간단한 프런트매터로 정의합니다. 참고: 프런트매터의 name이 파일 이름과 정확히 일치해야 합니다(대소문자 구분).

> **중요**: 출력 스타일은 Claude Code 내부의 시스템 프롬프트를 수정해, 소프트웨어 엔지니어링에 초점을 맞춘 부분을 덜어 내고 소프트웨어 엔지니어링을 넘어선 여러분의 사용 사례에 더 큰 통제권을 줍니다.

> **SDK 설정 참고**: (기능 4에서 다루는) 슬래시 명령과 마찬가지로 출력 스타일은 파일 시스템의 `.claude/output-styles/`에 저장됩니다. SDK가 이 파일들을 불러오려면 `ClaudeAgentOptions`에 `setting_sources=["project"]`를 **반드시** 포함해야 합니다. `settings` 파라미터는 *어떤* 스타일을 쓸지 알려 주지만, 스타일 정의를 실제로 *불러오려면* `setting_sources`가 필요합니다. 이 요구 사항은 뒤쪽 절을 디버깅하다 확인된 것으로, 파일 시스템 기반 설정 전반에 적용됩니다.

In [4]:
messages_executive = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "executive"}',
        # IMPORTANT: setting_sources must include "project" to load output styles from .claude/output-styles/
        # Without this, the SDK does NOT load filesystem settings (output styles, slash commands, etc.)
        setting_sources=["project"],
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_executive.append(msg)

messages_technical = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "technical"}',
        setting_sources=["project"],
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_technical.append(msg)

🤖 Thinking...
🤖 Thinking...


In [5]:
# Display executive style response
display_agent_response(messages_executive)

In [6]:
# Technical output style - detailed, implementation-focused
display_agent_response(messages_technical, title="Technical Style")

### 기능 3: 플랜 모드 — 실행 없는 전략 수립

**무엇**: 플랜 모드는 어떤 행동도 하지 않고 상세한 실행 계획을 세우도록 에이전트에 지시합니다. 에이전트는 요구 사항을 분석하고, 해결책을 제안하고, 단계를 정리하지만 파일을 수정하거나 명령을 실행하거나 무언가를 바꾸지는 않습니다.

**왜**: 복잡한 작업은 사전 계획으로 오류를 줄이고, 검토를 가능하게 하며, 협업을 개선할 수 있습니다. 계획 단계를 거치면 에이전트가 실행 내내 따라갈 하나의 실이 생깁니다.

**어떻게**: `permission_mode="plan"`으로 설정하기만 하면 됩니다

**계획 보존**: 계획은 검토와 의사결정에 가치 있는 산출물이므로, 이를 붙잡아 지속적인 마크다운 파일로 저장하는 방법을 보여 드립니다. 실행을 승인하기 전에 이해관계자가 계획을 검토할 수 있게 됩니다.

> 참고: 이 기능은 Claude Code에서 빛을 발하지만 SDK를 쓰는 헤드리스 애플리케이션에는 아직 완전히 맞춰지지 않았습니다. 구체적으로, 에이전트가 대화형 모드에서만 의미 있는 `ExitPlanMode()` 도구를 호출하려 합니다. 이 경우 `continue_conversation=True`로 후속 질의를 보내면 에이전트가 맥락 안에서 계획을 실행합니다.

In [7]:
# =============================================================================
# Plan Mode Helper Functions
# =============================================================================
# These utilities handle the various ways an agent might output its plan.
# Since agents can output plans via direct text, Write tool, or Claude's
# internal plan directory, we need robust extraction from multiple sources.

import glob as glob_module
import os
import re
from datetime import datetime
from pathlib import Path
from typing import Any


def extract_plan_from_xml(text: str | None, min_length: int = 200) -> str | None:
    """
    Extract content between <plan> tags from text.

    Args:
        text: The text to search for plan content
        min_length: Minimum character count for valid plan (prevents empty matches)

    Returns:
        Extracted plan content, or None if not found/too short
    """
    if not text:
        return None
    match = re.search(r"<plan>(.*?)</plan>", text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        if len(extracted) > min_length:
            return extracted
    return None


def extract_plan_from_messages(
    plan_content: list[str], min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Try to extract plan from captured message stream content.

    Args:
        plan_content: List of text blocks captured during streaming
        min_fallback_length: Minimum length for fallback (no XML tags)

    Returns:
        Tuple of (plan_text, source_description)
    """
    combined_text = "\n\n".join(plan_content)

    # First try: XML tags
    plan = extract_plan_from_xml(combined_text)
    if plan:
        return plan, "message stream"

    # Fallback: Use raw content if substantial
    if len(combined_text.strip()) > min_fallback_length:
        return combined_text.strip(), "full message content (fallback)"

    return None, None


def extract_plan_from_write_tool(
    write_contents: list[str], min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Try to extract plan from captured Write tool calls.

    Args:
        write_contents: List of content strings from Write tool calls
        min_fallback_length: Minimum length for fallback (no XML tags)

    Returns:
        Tuple of (plan_text, source_description)
    """
    for content in write_contents:
        # Try XML extraction first
        plan = extract_plan_from_xml(content)
        if plan:
            return plan, "Write tool capture"

        # Fallback: substantial content without tags
        if content and len(content.strip()) > min_fallback_length:
            return content.strip(), "Write tool capture (no XML tags)"

    return None, None


def extract_plan_from_claude_dir(
    max_age_seconds: int = 300, min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Check Claude's internal plan directory for recently created plans.

    Args:
        max_age_seconds: Maximum age of plan file to consider (default: 5 minutes)
        min_fallback_length: Minimum length for fallback (no XML tags)

    Returns:
        Tuple of (plan_text, source_description)
    """
    claude_plans_dir = os.path.expanduser("~/.claude/plans")

    if not os.path.exists(claude_plans_dir):
        return None, None

    # Find most recent plan file
    plan_files = sorted(
        glob_module.glob(os.path.join(claude_plans_dir, "*.md")),
        key=os.path.getmtime,
        reverse=True,
    )

    if not plan_files:
        return None, None

    most_recent = plan_files[0]
    file_age = datetime.now().timestamp() - os.path.getmtime(most_recent)

    if file_age > max_age_seconds:
        return None, None

    with open(most_recent) as f:
        content = f.read()

    filename = os.path.basename(most_recent)

    # Try XML extraction first
    plan = extract_plan_from_xml(content)
    if plan:
        return plan, f"Claude plan file ({filename})"

    # Fallback: substantial content without tags
    if len(content.strip()) > min_fallback_length:
        return content.strip(), f"Claude plan file ({filename}, no XML tags)"

    return None, None


def save_plan_to_file(
    plan_content: str,
    plan_source: str,
    model_name: str,
    prompt_summary: str,
    output_dir: str = "chief_of_staff_agent/plans",
    title: str = "Agent Plan: Engineering Restructure for AI Focus",
) -> Path:
    """
    Save extracted plan to a timestamped markdown file.

    Args:
        plan_content: The plan text to save
        plan_source: Description of where plan was extracted from
        model_name: The model used to generate the plan
        prompt_summary: Brief description of the original prompt
        output_dir: Directory to save plan files
        title: Title for the plan document

    Returns:
        Path to the saved plan file
    """
    plans_dir = Path(output_dir)
    plans_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    plan_file = plans_dir / f"plan_{timestamp}.md"

    with open(plan_file, "w") as f:
        f.write(f"# {title}\n\n")
        f.write(f"**Created:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"**Prompt:** {prompt_summary}\n")
        f.write(f"**Model:** {model_name}\n")
        f.write(f"**Plan Source:** {plan_source}\n\n")
        f.write("---\n\n")
        f.write(plan_content)
        f.write("\n\n---\n\n")
        f.write("*This plan was generated in plan mode and has not been executed.*\n")

    return plan_file


def capture_message_content(
    msg: Any,
    plan_content: list[str],
    write_tool_content: list[str],
    write_tool_paths: list[str],
) -> None:
    """
    Process a streaming message and capture relevant plan content.

    This function extracts content from three potential sources:
    1. Text blocks in message content
    2. Write tool call parameters
    3. Final result attribute

    Args:
        msg: The message object from the agent stream
        plan_content: List to append text content to
        write_tool_content: List to append Write tool content to
        write_tool_paths: List to append Write tool file paths to
    """
    # Source 1: Text blocks from message content
    if hasattr(msg, "content"):
        for block in msg.content:
            if hasattr(block, "text"):
                plan_content.append(block.text)

            # Source 2: Write tool calls
            if hasattr(block, "type") and block.type == "tool_use":
                if hasattr(block, "name") and block.name == "Write":
                    if hasattr(block, "input") and isinstance(block.input, dict):
                        if "content" in block.input:
                            write_tool_content.append(block.input["content"])
                        if "file_path" in block.input:
                            write_tool_paths.append(block.input["file_path"])

    # Source 3: Final result
    if hasattr(msg, "result") and msg.result:
        plan_content.append(msg.result)


print("✅ Plan Mode helper functions loaded")

✅ Plan Mode helper functions loaded


In [8]:
# =============================================================================
# Plan Mode Configuration
# =============================================================================

# Note: MODEL is defined in cell-0 as "claude-opus-4-6"
# Opus excels at complex planning tasks

# The prompt is carefully crafted to:
# 1. Provide explicit context (since Opus prefers explicit information)
# 2. Request XML-tagged output for reliable extraction
# 3. Prevent file-writing so we can capture the plan programmatically

PLAN_PROMPT = """Restructure our engineering team for AI focus.

**CONTEXT (from CLAUDE.md):**
You are the Chief of Staff for TechStart Inc, a 50-person B2B SaaS startup that raised $10M Series A.
- Current engineering team: 25 people (Backend: 12, Frontend: 8, DevOps: 5)
- Monthly burn rate: ~$500K, Runway: 20 months
- Senior Engineer compensation: $180K-$220K + equity

**CRITICAL OUTPUT INSTRUCTIONS:**

1. **DO NOT use the Write tool** - Output your plan directly in your response text
2. **DO NOT save to any files** - I will handle saving the plan myself
3. **Wrap your ENTIRE plan inside `<plan> </plan>` XML tags** in your response

**Required Format:**
<plan>
[Your complete restructuring plan here - include all sections, timelines, budgets, and recommendations]
</plan>

**IMPORTANT:**
- The plan content MUST appear directly in your response between the XML tags
- Do NOT use Write, Edit, or any file-saving tools
- You may research and analyze before outputting, but the final plan must be in your response text
- Include: team structure, hiring recommendations, timeline, budget impact, and success metrics
- Use the company context provided above - do NOT ask clarifying questions"""

print(f"📋 Plan Mode configured with model: {MODEL}")
print(f"📝 Prompt length: {len(PLAN_PROMPT):,} characters")

📋 Plan Mode configured with model: claude-opus-4-6
📝 Prompt length: 1,180 characters


In [9]:
# =============================================================================
# Execute Plan Mode Agent
# =============================================================================
# Run the agent with plan mode enabled. The agent will create a detailed plan
# but won't execute any actions. We capture content from multiple sources
# to handle different agent behaviors.

# Initialize capture lists
messages = []
plan_content = []  # Text from message stream
write_tool_content = []  # Content from Write tool calls
write_tool_paths = []  # Paths from Write tool calls

# Run the agent in plan mode
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        permission_mode="plan",
        cwd="chief_of_staff_agent",
    )
) as agent:
    await agent.query(PLAN_PROMPT)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

        # Capture content from this message
        capture_message_content(msg, plan_content, write_tool_content, write_tool_paths)

print(f"\n✅ Agent completed. Captured {len(plan_content)} content blocks.")

🤖 Thinking...
🤖 Using: ExitPlanMode()
✓ Tool completed
🤖 Thinking...

✅ Agent completed. Captured 3 content blocks.


In [10]:
# =============================================================================
# Extract and Save the Plan
# =============================================================================
# Try multiple sources in priority order to find the plan content.
# This handles different agent behaviors robustly.

final_plan = None
plan_source = None

# Priority 1: Message stream (preferred - direct from agent response)
final_plan, plan_source = extract_plan_from_messages(plan_content)

# Priority 2: Write tool captures (if agent saved despite instructions)
if not final_plan and write_tool_content:
    final_plan, plan_source = extract_plan_from_write_tool(write_tool_content)

# Priority 3: Claude's internal plan directory (safety net)
if not final_plan:
    final_plan, plan_source = extract_plan_from_claude_dir()

# Report results
if final_plan:
    print(f"✅ Plan extracted from: {plan_source}")
    print(f"   Plan length: {len(final_plan):,} characters")

    # Save to file
    plan_file = save_plan_to_file(
        plan_content=final_plan,
        plan_source=plan_source,
        model_name=MODEL,
        prompt_summary="Restructure our engineering team for AI focus.",
    )
    print(f"\n📁 Plan saved to: {plan_file}")
else:
    error_msg = "Could not extract plan content from any source!\n"
    error_msg += "   Sources checked: message stream, Write tool, ~/.claude/plans/"
    if write_tool_paths:
        error_msg += f"\n   Write tool attempted to save to: {write_tool_paths}"
    print(f"❌ ERROR: {error_msg}")
    raise RuntimeError(f"Plan extraction failed: {error_msg}")

✅ Plan extracted from: message stream
   Plan length: 6,783 characters

📁 Plan saved to: chief_of_staff_agent/plans/plan_20251204_152737.md


In [11]:
# Display the plan result with styled HTML
display_agent_response(messages, title="Engineering Restructure Plan")

#### 저장된 계획 실행하기

위에서 말했듯 에이전트는 계획을 만든 뒤 멈춥니다. 저장된 계획 파일은 이해관계자를 위한 검토 산출물이 됩니다.

**검토 후 계획을 실행하려면:**
1. `chief_of_staff_agent/plans/plan_*.md`에 저장된 계획을 검토하세요
2. 승인되면 `continue_conversation=True`로 새 질의를 보내고 `permission_mode="plan"`을 제거해 실행하세요

이 흐름은 "계획 → 검토 → 승인 → 실행" 주기를 가능하게 하며, 조직 개편이나 주요 인프라 변경 같은 중대한 결정에 안성맞춤입니다.

#### 계획 보존이 동작하는 방식

위 코드에서는 플랜 모드 에이전트가 계획을 내놓는 여러 방식을 모두 처리하는 **다중 출처 계획 포착 메커니즘**을 구현했습니다.

**난제:**
`permission_mode="plan"`을 쓰면 에이전트가 계획을 여러 방식으로 내놓을 수 있습니다.
1. 메시지 스트림의 **직접 텍스트 출력**(이상적인 경우)
2. `~/.claude/plans/`에 저장하는 **Write 도구**(Claude 내부 계획 시스템)
3. 커스텀 경로에 저장하는 **Write 도구**

우리의 포착 메커니즘은 **우선순위 기반 폴백 시스템**으로 세 경우를 모두 처리합니다.

**출처 우선순위(순서대로):**

1. **메시지 스트림**(선호)
   - 스트리밍 중 `msg.content`의 텍스트 블록을 포착
   - `<plan></plan>` XML 태그 사이의 내용을 추출
   - 내용이 응답에서 곧바로 오므로 가장 깔끔한 방식

2. **Write 도구 포착**
   - 메시지 스트림에서 Write 도구 호출을 감시
   - 기록되는 `content` 파라미터를 추출
   - 프롬프트 지시에도 불구하고 에이전트가 저장하기로 결정했을 때 유용

3. **Claude 내부 계획 디렉터리**
   - `~/.claude/plans/`에서 최근(5분 이내) 생성된 계획 파일 확인
   - 가장 최근 파일을 읽어 내용 추출
   - 다른 방법이 실패했을 때의 안전망

4. **전체 내용 폴백**
   - XML 태그를 찾지 못했지만 충분한 내용(500자 초과)이 있으면 그대로 사용
   - 부분 정보를 보존하면서 빈 계획 파일이 만들어지는 것을 방지

**핵심 구현 세부 사항:**

```python
def extract_plan_from_text(text):
    """Extract content between <plan> tags, return None if not found or empty."""
    match = re.search(r'<plan>(.*?)</plan>', text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        # Validate minimum content length (a real plan should be substantial)
        if len(extracted) > 200:
            return extracted
    return None
```

**내용 검증이 중요한 이유:**
- 이전 버전에서는 추출이 "성공"했는데 내용이 없어 빈 계획 파일이 만들어질 수 있었습니다
- 이제 XML 태그로 감싼 내용은 최소 200자를 요구합니다
- 정규식이 비어 있거나 사소한 내용에 매칭되는 거짓 양성을 막아 줍니다

**직접 출력을 위한 프롬프트 엔지니어링:**
프롬프트는 에이전트에 명시적으로 지시합니다.
- **Write 도구를 쓰지 말 것** — 파일 시스템으로 우회하는 것을 방지
- **응답에 직접 출력할 것** — 내용이 메시지 스트림을 통해 흐르도록 보장
- **XML 태그를 쓸 것** — 장황할 수 있는 응답에서 깔끔하게 추출 가능

이 방식이 주는 것:
- **신뢰성**: 에이전트의 동작과 무관하게 계획이 포착됩니다
- **투명성**: 저장된 파일에 어떤 출처를 썼는지 표시됩니다
- **감사 기록**: 타임스탬프와 출처 메타데이터가 담긴 모든 계획의 이력
- **디버깅**: 추출 실패 시 명확한 오류 메시지

저장된 계획을 살펴보겠습니다:

In [15]:
# Display the saved plan with markdown rendering
from IPython.display import Markdown, display

# Show the plan with proper markdown formatting
display(Markdown(f"## 📋 Saved Plan Preview\n\n{final_plan}"))

print(f"\n📁 Full plan with metadata saved to: {plan_file}")

## 📋 Saved Plan Preview

# TechStart Inc. Engineering Team Restructuring Plan for AI Focus

## Executive Summary
Restructure the 25-person engineering team to build AI/ML capabilities while maintaining core product development. This plan balances immediate hiring needs with internal upskilling and strategic reorganization over a 12-month period.

---

## Current State Assessment

| Team | Headcount | Current Focus |
|------|-----------|---------------|
| Backend | 12 | Core product, APIs, infrastructure |
| Frontend | 8 | Web/mobile interfaces |
| DevOps | 5 | CI/CD, cloud infrastructure |
| **Total** | **25** | |

**Key Constraints:**
- Monthly burn: ~$500K | Runway: 20 months
- Senior Engineer comp: $180K-$220K + equity
- Series A stage - need to show growth metrics

---

## Proposed Future State (Month 12)

### New Team Structure

| Team | Current | Future | Change |
|------|---------|--------|--------|
| Backend/Core | 12 | 8 | -4 |
| Frontend | 8 | 6 | -2 |
| DevOps/MLOps | 5 | 6 | +1 |
| **AI/ML Platform** | 0 | 5 | +5 |
| **AI Product** | 0 | 4 | +4 |
| **Total** | **25** | **29** | **+4** |

### New Teams Created

**1. AI/ML Platform Team (5 engineers)**
- Focus: Infrastructure, model training pipelines, ML tooling
- Composition: 2 new hires (ML Engineers) + 2 internal transfers (Backend) + 1 internal (DevOps)
- Lead: New hire - Senior ML Engineer ($200-220K)

**2. AI Product Team (4 engineers)**
- Focus: AI features, integrations, user-facing ML applications
- Composition: 2 new hires (ML/AI specialists) + 1 internal (Backend) + 1 internal (Frontend)
- Lead: Promoted internal senior engineer + AI upskilling

---

## Hiring Plan

### New Positions (6 total new hires)

| Role | Priority | Timeline | Comp Range | Notes |
|------|----------|----------|------------|-------|
| Senior ML Engineer (Lead) | P0 | Month 1-2 | $200-220K | Team lead, architecture |
| ML Engineer | P0 | Month 2-3 | $180-200K | Platform focus |
| ML Engineer | P1 | Month 3-4 | $180-200K | Product focus |
| AI/ML Engineer | P1 | Month 4-5 | $170-190K | Generalist |
| MLOps Engineer | P2 | Month 5-6 | $160-180K | DevOps + ML |
| Junior ML Engineer | P2 | Month 6-8 | $130-150K | Growth hire |

**Total New Hiring Cost:** ~$1.02-1.14M annually

### Internal Transfers & Upskilling (4 engineers)

| From Team | # Engineers | New Role | Training Investment |
|-----------|-------------|----------|---------------------|
| Backend | 3 | ML Platform/Product | $15K each (courses, certs) |
| Frontend | 1 | AI Product (UI/UX) | $10K (AI tools training) |

**Upskilling Budget:** ~$55K total

---

## Implementation Timeline

### Phase 1: Foundation (Months 1-3)
- [ ] Hire Senior ML Engineer (Team Lead) - **Critical first hire**
- [ ] Identify 4 internal transfer candidates based on interest/aptitude
- [ ] Begin ML upskilling program for transfer candidates
- [ ] Set up ML infrastructure foundations (MLOps engineer involvement)
- [ ] Define AI product roadmap with Product team

### Phase 2: Team Formation (Months 4-6)
- [ ] Complete core AI team hiring (4 of 6 hires)
- [ ] Officially launch AI/ML Platform team
- [ ] Internal transfers complete bootcamp/training
- [ ] First AI feature POC delivered
- [ ] MLOps practices integrated into DevOps workflow

### Phase 3: Scaling (Months 7-9)
- [ ] Complete remaining hires
- [ ] AI Product team fully operational
- [ ] First AI feature in production
- [ ] Cross-team collaboration patterns established
- [ ] Model monitoring and observability in place

### Phase 4: Optimization (Months 10-12)
- [ ] Team velocity optimization
- [ ] Evaluate team composition effectiveness
- [ ] Plan for next growth phase
- [ ] Document AI development best practices

---

## Budget Impact Analysis

### Monthly Cost Changes

| Category | Current | Month 6 | Month 12 |
|----------|---------|---------|----------|
| Engineering Salaries | ~$375K | ~$430K | ~$485K |
| Training/Upskilling | $0 | $9K | $2K |
| AI Tools/Infrastructure | ~$5K | ~$20K | ~$35K |
| Recruiting Costs | Variable | ~$40K | ~$15K |
| **Monthly Total** | **~$380K** | **~$499K** | **~$537K** |

### Annual Impact Summary
- **Year 1 Additional Investment:** ~$900K-1.1M
- **New Monthly Burn (Month 12):** ~$537K (+$37K from engineering growth)
- **Runway Impact:** Reduces to ~17 months (still healthy for Series A)

### ROI Considerations
- AI features typically command 20-40% pricing premium in B2B SaaS
- Competitive differentiation in market
- Potential for AI-driven operational efficiencies

---

## Risk Mitigation

| Risk | Likelihood | Impact | Mitigation |
|------|------------|--------|------------|
| Senior ML hire takes >3 months | Medium | High | Engage specialized recruiters early; consider contractor bridge |
| Internal transfers struggle with ML | Low | Medium | Rigorous selection process; extended training runway |
| AI projects don't deliver value | Medium | High | Start with high-impact, lower-complexity features |
| Team culture friction | Low | Medium | Integrate teams gradually; shared goals and rituals |
| Budget overrun | Medium | Medium | Phase hiring based on runway checks quarterly |

---

## Success Metrics

### 6-Month Milestones
- [ ] AI/ML Platform team fully staffed (5 engineers)
- [ ] First AI-powered feature in beta
- [ ] 4 internal engineers completed ML certification
- [ ] ML infrastructure supporting model training/deployment

### 12-Month Milestones
- [ ] 2+ AI features in production
- [ ] AI team velocity matches core team benchmarks
- [ ] Customer NPS improvement attributable to AI features
- [ ] 15%+ of product roadmap is AI-focused
- [ ] Retention of transferred engineers >90%

### KPIs to Track
- Time-to-hire for ML roles
- AI feature adoption rates
- Model performance metrics (latency, accuracy)
- Engineering velocity (story points/sprint) across teams
- Employee satisfaction scores (especially transfers)

---

## Immediate Next Steps (Week 1-2)

1. **Executive alignment** - Present plan to leadership for approval
2. **Recruiter engagement** - Brief recruiting on Senior ML Engineer search
3. **Internal survey** - Gauge interest in AI/ML roles among current engineers
4. **Budget approval** - Finance sign-off on increased burn rate
5. **Infrastructure assessment** - DevOps audit of ML infrastructure needs

---

## Organizational Chart (Future State)

```
VP Engineering
├── Backend/Core Team (8)
│   └── 2 squads × 4 engineers
├── Frontend Team (6)
│   └── 2 squads × 3 engineers
├── DevOps/MLOps Team (6)
│   └── 4 DevOps + 2 MLOps
├── AI/ML Platform Team (5) [NEW]
│   └── Lead + 4 engineers
└── AI Product Team (4) [NEW]
    └── Lead + 3 engineers
```

---

*Plan prepared for TechStart Inc. Series A stage (~$10M raised, 20-month runway)*
*Recommended review: Quarterly budget and hiring progress checkpoints*


📁 Full plan with metadata saved to: chief_of_staff_agent/plans/plan_20251204_152737.md


## 고급 기능

### 기능 4: 커스텀 슬래시 명령

> 참고: 슬래시 명령은 새로운 에이전트 역량이 아니라 사용자를 위한 편의 문법입니다

**무엇**: 커스텀 슬래시 명령은 사용자가 짧은 문법(예: `/budget-impact`)으로 실행할 수 있는 미리 정의된 프롬프트 템플릿입니다. 에이전트 역량이 아니라 **사용자 대면 단축키**입니다. 잘 다듬어진 전체 프롬프트로 펼쳐지는 키보드 단축키라고 생각하세요.

**왜**: Chief of Staff는 반복되는 경영진 질문을 다루게 됩니다. 사용자가 복잡한 프롬프트를 매번 입력하는 대신 이미 검증된 프롬프트를 쓸 수 있습니다. 일관성과 표준화가 좋아집니다.

**어떻게**:
- `.claude/commands/`에 마크다운 파일을 정의하세요. 예로 `.claude/commands/slash-command-test.md`를 만들어 두었습니다. 명령이 어떻게 정의되는지 보세요. 두 필드(name, description)로 된 프런트매터와, 질의로 전달된 인자를 포함할 수 있는 확장 프롬프트입니다.
- `$ARGUMENTS`(전체 인자 문자열)나 `$1`, `$2` 등(위치 인자)으로 프롬프트에 파라미터를 넣을 수 있습니다
- 사용자는 프롬프트에서 슬래시 명령을 사용합니다

> **중요한 SDK 설정**: SDK를 사용할 때 슬래시 명령이 동작하려면 `ClaudeAgentOptions`에 `setting_sources=["project"]`를 **반드시** 설정해야 합니다. 기본적으로 SDK는 격리 모드로 동작하며 파일 시스템 설정(슬래시 명령, CLAUDE.md, 서브에이전트, 훅 등)을 불러오지 않습니다. 이들이 자동으로 적재되는 대화형 Claude Code 사용과 다른 점입니다.

In [16]:
# User types: "/slash-command-test this is a test"
# -> behind the scenes EXPANDS to the prompt in .claude/commands/slash-command-test.md
# In this case the expanded prompt says to simply reverse the sentence word wise

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        # IMPORTANT: setting_sources must include "project" to load slash commands from .claude/commands/
        # Without this, the SDK does NOT load filesystem settings (slash commands, CLAUDE.md, etc.)
        setting_sources=["project"],
    )
) as agent:
    await agent.query("/slash-command-test this is a test")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...


In [17]:
display_agent_response(messages, title="Slash Command Result")

### 기능 5: 훅 — 자동화된 결정론적 동작

**무엇**: 훅은 여러 이벤트 중에서도 특정 도구 호출 전(pre)이나 후(post)에 자동으로 실행되도록 설정할 수 있는 Python 스크립트입니다. 훅은 **결정론적으로** 실행되므로 검증과 감사 기록에 안성맞춤입니다.

**왜**: 에이전트에 안전장치를 두고 싶거나(예: 위험한 작업 방지) 감사 기록을 남기고 싶은 상황을 떠올려 보세요. 훅은 에이전트가 작업을 해내기에 충분한 자유를 주면서도 안전하게 행동하도록 보장하는 데 이상적입니다.

**어떻게**:
- `.claude/hooks/`에 훅 스크립트를 정의합니다 → 훅이 발동할 때 실행될 _동작_
- `.claude/settings.local.json`에 훅 설정을 정의합니다 → 훅이 _언제_ 발동할지
- 이 경우 훅은 특정 도구 호출(Bash, Write, Edit)을 감시하도록 설정되어 있습니다
- 해당 도구가 호출되면 도구가 끝난 뒤 훅 스크립트가 실행됩니다(PostToolUse)

> **SDK 설정 참고**: `.claude/settings.local.json`에 설정한 훅에는 `setting_sources=["project", "local"]`이 필요합니다. SDK는 세 가지 설정 출처를 구분합니다.
> - `"project"` → `.claude/settings.json`(버전 관리, 팀 공유)
> - `"local"` → `.claude/settings.local.json`(gitignore, 훅 같은 로컬 설정)
> - `"user"` → `~/.claude/settings.json`(전역 사용자 설정)
>
> 우리 훅은 `settings.local.json`에 있으므로 `setting_sources`에 `"local"`을 포함해야 합니다.

**예제: 컴플라이언스를 위한 보고서 추적**

감사와 컴플라이언스를 위해 재무 보고서에 대한 Write/Edit 작업을 기록하는 훅입니다.
훅은 `chief_of_staff_agent/.claude/hooks/report-tracker.py`에 정의되어 있고, 이를 강제하는 로직은 `chief_of_staff_agent/.claude/settings.local.json`에 있습니다:


```json
"hooks": {
  "PostToolUse": [
    {
      "matcher": "Write",
      "hooks": [
        {
          "type": "command",
          "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
        }
      ]
    },
    {
      "matcher": "Edit",
      "hooks": [
        {
          "type": "command",
          "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
        }
      ]
    }
  ]
}
```

In [18]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        allowed_tools=["Bash", "Write", "Edit", "MultiEdit"],
        # IMPORTANT: setting_sources must include BOTH "project" AND "local" to load hooks
        # - "project" loads .claude/settings.json (shared settings, CLAUDE.md, slash commands)
        # - "local" loads .claude/settings.local.json (where hooks are configured)
        setting_sources=["project", "local"],
    )
) as agent:
    await agent.query(
        "Create a quick Q2 financial forecast report with our current burn rate and runway projections. Save it to our /output_reports folder."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# The hook will track this in audit/report_history.json
display_agent_response(messages, title="Q2 Financial Forecast")

🤖 Using: Bash()
🤖 Using: Bash()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


Metric,Value
Cash in Bank,$10M
Current Runway,20 months (until Sept 2025)
Monthly Burn,$500K gross / ~$260K net
Q2 Total Net Burn,$740K
ARR,$2.4M (15% MoM growth)


이제 `./chief_of_staff_agent/audit/report_history.json`으로 가 보면, 에이전트가 보고서를 만들거나 수정한 기록이 남아 있는 것을 볼 수 있습니다. 생성된 보고서 자체는 `./chief_of_staff_agent/output_reports/`에 있습니다.

### 기능 6: Task 도구를 통한 서브에이전트

**무엇**: Task 도구를 사용하면 에이전트가 전문적인 작업을 다른 서브에이전트에 위임할 수 있습니다. 각 서브에이전트는 자기만의 지시문, 도구, 전문성을 갖습니다.

**왜**: 서브에이전트를 더하면 여러 가능성이 열립니다.
1. 전문화: 각 서브에이전트가 자기 영역의 전문가입니다
2. 분리된 컨텍스트: 서브에이전트는 자기만의 대화 기록과 도구를 갖습니다
3. 병렬화: 여러 서브에이전트가 서로 다른 측면을 동시에 처리할 수 있습니다

**어떻게**:
- allowed_tools에 `"Task"`를 추가합니다
- 시스템 프롬프트로 작업을 어떻게 위임할지 에이전트에 지시합니다(CLAUDE.md에 더 일반적으로 정의할 수도 있습니다)
- `.claude/agents/`에 에이전트마다 마크다운 파일을 만듭니다. 예로 `.claude/agents/financial-analyst.md`를 확인해 보세요. 이렇게 쉽고 직관적인 마크다운 파일만으로 (서브)에이전트를 정의할 수 있습니다. 세 필드(name, description, tools)로 된 프런트매터와 시스템 프롬프트입니다. description은 메인 Chief of Staff 에이전트가 각 서브에이전트를 언제 호출할지 판단하는 데 유용합니다.

**시각화 개선**: `print_activity()`와 `visualize_conversation()` 유틸리티를 개선해 서브에이전트 동작을 명확히 보여 줍니다.
- 🚀 는 서브에이전트에 위임되는 시점을 표시합니다(서브에이전트 이름과 함께)
- 📎 는 서브에이전트가 사용하는 도구를 표시합니다(시각적 계층을 위해 들여쓰기)
- 구분선이 서브에이전트 위임과 완료 경계를 분명히 표시합니다
- 대화 타임라인에 작업 설명과 프롬프트가 표시됩니다

In [19]:
# Reset the subagent tracking context before starting a new query
# This ensures clean state for activity display
reset_activity_context()

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        allowed_tools=["Task"],  # this enables our Chief agent to invoke subagents
        system_prompt="Delegate financial questions to the financial-analyst subagent. Do not try to answer these questions yourself.",
        cwd="chief_of_staff_agent",
        setting_sources=["project", "local"],
    )
) as agent:
    await agent.query("Should we hire 5 engineers? Analyze the financial impact.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages, title="Hiring Impact Analysis")

🤖 Thinking...
🚀 Delegating to subagent: financial-analyst
   └─ Task: Analyze 5 engineer hiring impact
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   📎 [financial-analyst] Using: Read()
   📎 [financial-analyst] Using: Read()
   ✓ Tool completed
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Using: Read()
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Thinking...


Metric,Current,After 5 Hires,Change
Monthly Burn,"$500,000","$575,833",+15.2%
Runway,20 months,17.1 months,-2.9 months
Headcount,50,55,+10%
Engineering %,50%,54.5%,+4.5%
Scenario,Mix,New Monthly Burn,New Runway
A (Recommended),3 Senior + 2 Junior,"$575,833",17.1 months
B,5 Senior,"$591,665",16.6 months
C,2 Senior + 3 Junior,"$567,917",17.3 months


In [20]:
visualize_conversation(messages)

Metric,Current,After 5 Hires,Change
Monthly Burn,"$500,000","$575,833",+15.2%
Runway,20 months,17.1 months,-2.9 months
Headcount,50,55,+10%
Engineering %,50%,54.5%,+4.5%
Scenario,Mix,New Monthly Burn,New Runway
A (Recommended),3 Senior + 2 Junior,"$575,833",17.1 months
B,5 Senior,"$591,665",16.6 months
C,2 Senior + 3 Junior,"$567,917",17.3 months


여기서 메인 에이전트가 서브에이전트를 쓰기로 하면 다음과 같이 진행됩니다.
  1. 다음과 같은 파라미터로 Task 도구를 호출합니다:
  ```json
    {
      "description": "Analyze hiring impact",
      "prompt": "Analyze the financial impact of hiring 5 engineers...",
      "subagent_type": "financial-analyst"
    }
  ```
  2. Task 도구가 별도의 컨텍스트에서 서브에이전트를 실행합니다
  3. 결과를 메인 Chief of Staff 에이전트에 돌려주어 처리를 이어 갑니다

## 전부 합쳐 보기

이제 지금까지 본 것을 모두 합쳐 보겠습니다. 시니어 엔지니어 3명 채용의 재무적 영향을 판단하고 그 통찰을 `output_reports/hiring_decision.md`에 쓰라고 에이전트에 요청합니다. 위에서 본 모든 기능이 동원됩니다.
- **Bash 도구**: `hiring_impact.py` 스크립트를 실행해 신규 채용의 영향을 판단합니다
- **메모리**: 디렉터리의 `CLAUDE.md`를 맥락으로 읽어 현재 예산, 런웨이, 매출 등 관련 정보를 파악합니다
- **출력 스타일**: `chief_of_staff_agent/.claude/output-styles`에 정의된 여러 출력 스타일
- **커스텀 슬래시 명령**: `chief_of_staff_agent/.claude/commands`에 정의된 전체 프롬프트로 펼쳐지는 단축키 `/budget-impact` 사용
- **서브에이전트**: `/budget_impact` 명령이 Chief of Staff 에이전트로 하여금 `chief_of_staff_agent/.claude/agents`에 정의된 financial-analyst 서브에이전트를 호출하도록 안내합니다
- **훅**: 훅은 `chief_of_staff_agent/.claude/hooks`에 정의되고 `chief_of_staff_agent/.claude/settings.local.json`에 설정됩니다
    - 에이전트 중 하나가 재무 보고서를 갱신하면, 훅이 그 편집/쓰기 활동을 `chief_of_staff_agent/audit/report_history.json` 로그 파일에 기록합니다
    - financial-analyst 서브에이전트가 `hiring_impact.py` 스크립트를 호출하면 `chief_of_staff_agent/audit/tool_usage_log.json` 로그 파일에 기록됩니다

- **플랜 모드**: 어떤 행동을 하기 전에 Chief of Staff가 승인받을 계획을 먼저 세우기를 원한다면 아래 주석 처리된 줄의 주석을 해제하세요

바로 쓸 수 있도록, 이전 노트북에서 했던 것처럼 에이전트 루프를 파이썬 파일에 담아 두었습니다. `chief_of_staff_agent` 하위 디렉터리의 agent.py 파일을 확인해 보세요.

정리하면, `send_query()` 함수는 네 개의 파라미터(prompt, continue_conversation, permission_mode, output_style)를 받고, 나머지(시스템 프롬프트, 최대 턴 수, 허용 도구, 작업 디렉터리)는 모두 에이전트 파일에 설정되어 있습니다.

이 모든 것이 어떻게 맞물리는지 더 잘 보려면 [Claude가 만들어 준 흐름도와 아키텍처 다이어그램](./chief_of_staff_agent/flow_diagram.md)을 확인해 보세요.

In [21]:
from chief_of_staff_agent.agent import send_query

reset_activity_context()

result, messages = await send_query(
    "/budget-impact hiring 3 senior engineers. Save your insights by updating the 'hiring_decision.md' file in /output_reports or creating a new file there",
    # permission_mode="plan", # Enable this to use planning mode
    output_style="executive",
)

🤖 Thinking...
🤖 Using: Glob()
🤖 Using: Glob()
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Task()
✓ Tool completed
🤖 Using: Bash()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


In [22]:
visualize_conversation(messages)

Metric,Current,Post-Hiring,Change
Monthly Gross Burn,$525K,$590K,+$65K (+12.4%)
Monthly Net Burn,$235K,$300K,+$65K (+27.7%)
Cash Runway,42.6 months,32.9 months,-9.7 months
Break-Even,-,November 2024,5 months


## 마무리

Claude Code SDK로 엔터프라이즈 수준 기능을 갖춘 정교한 멀티에이전트 시스템을 만드는 방법을 살펴봤습니다. Bash 도구를 통한 기본적인 스크립트 실행에서 시작해, CLAUDE.md를 통한 지속적 메모리, 청중별 커스텀 출력 스타일, 전략 수립을 위한 플랜 모드, 사용자 편의를 위한 슬래시 명령, 안전장치를 위한 컴플라이언스 훅, 그리고 전문 작업을 위한 서브에이전트 조율까지 고급 역량을 차례로 소개했습니다.

이 기능들을 결합해 복잡한 경영 의사결정 워크플로를 처리할 수 있는 AI Chief of Staff를 만들었습니다. 이 시스템은 재무 분석을 전문 서브에이전트에 위임하고, 훅으로 감사 기록을 유지하며, 이해관계자에 따라 소통 방식을 조정하고, 데이터에 근거한 실행 가능한 통찰을 제공합니다.

고급 에이전트 패턴과 멀티에이전트 오케스트레이션의 이 토대는 프로덕션 수준 엔터프라이즈 시스템을 만들 준비를 해 줍니다. 다음 노트북에서는 Model Context Protocol(MCP) 서버로 에이전트를 외부 서비스에 연결해 내장 도구를 훌쩍 넘어서는 역량으로 확장하는 방법을 살펴봅니다.

다음: [02_The_observability_agent.ipynb](02_The_observability_agent.ipynb) — MCP를 통해 커스텀 통합과 외부 데이터 출처로 에이전트를 확장하는 방법을 배웁니다.